# Maritime Workflow Performance Metrics

Visualization of pipeline performance across backends (PostGIS, GeoPackage) and graph modes (FINE, H3).

Data source: `../../scripts/benchmarks/MaritimeWorkflowPerformanceMetrics.csv`

### Figures
1. **Base Graph Creation** — by Backend
2. **Fine/H3 Graph Creation** — by Backend × Graph Mode
3. **Weighting & Enrichment** — by Backend (2× edge count on X axis)
4. **Pathfinding & Export** — by Backend (edge count; A* explores edges)
5. **Total Time** — by Backend × Graph Mode

In [1]:
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

PLOTLY_THEME = "plotly_dark"

# --- Color & Style Constants ---
BACKEND_COLORS = {"PostGIS": "#636EFA", "GeoPackage": "#EF553B"}
MODE_DASH = {"H3": "dash", "FINE": "solid"}

# Get project root for .env file loading
project_root = Path.cwd().parent.parent
assert (project_root / "src" / "nautical_graph_toolkit").exists()

# --- Load & Clean ---
csv_path = project_root / 'scripts' /'benchmarks' / 'MaritimeWorkflowPerformanceMetrics.csv'
df = pd.read_csv(csv_path)

time_cols = [
    "Base Graph Creation (s)",
    "Fine/H3 Graph Creation (s)",
    "Weighting & Enrichment (s)",
    "Pathfinding & Export (s)",
    "Total Time (s)",
]
for col in time_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Derived columns
df["edges_per_1M"] = df["Final Undirected Edges Count"] / 1_000_000
df["edges_2x_per_1M"] = df["Final Undirected Edges Count"] * 2 / 1_000_000

# Convert seconds → minutes
for col in time_cols:
    df[f"{col.rstrip(' (s)')} (min)"] = df[col] / 60

df.sort_values("edges_per_1M", inplace=True)

# Report data completeness
complete = df.dropna(subset=time_cols).shape[0]
partial = df.shape[0] - complete
print(f"Loaded {len(df)} workflow runs  |  PostGIS: {(df['Backend']=='PostGIS').sum()}  GeoPackage: {(df['Backend']=='GeoPackage').sum()}")
print(f"Edge range: {df['edges_per_1M'].min():.1f} – {df['edges_per_1M'].max():.1f}  (×1M)")
if partial:
    print(f"Complete: {complete}  |  Partial (incomplete pipeline): {partial}")
    partial_ids = df[df[time_cols].isna().any(axis=1)]["Workflow ID"].tolist()
    print(f"  Partial runs: {', '.join(partial_ids)}")

Loaded 45 workflow runs  |  PostGIS: 33  GeoPackage: 12
Edge range: 0.2 – 13.5  (×1M)
Complete: 39  |  Partial (incomplete pipeline): 6
  Partial runs: workflow_fine_sf_test_20260513_092531, workflow_fine_mia_kw2_20260514_214307, workflow_fine_mia_kw_20260514_212232, workflow_h3_sf_la_511_20260514_135154, workflow_h3_gal_arrival_511_20260515_183221, workflow_fine_la_west4_20260512_213401


## Figure 1 — Base Graph Creation
Time to create the base graph, grouped by backend.

In [2]:
y_col = "Base Graph Creation (min)"
valid = df.dropna(subset=[y_col])

fig = go.Figure()
for backend in ["PostGIS", "GeoPackage"]:
    subset = valid[valid["Backend"] == backend].sort_values("edges_per_1M")
    fig.add_trace(go.Scatter(
        x=subset["edges_per_1M"],
        y=subset[y_col],
        mode="markers",
        name=backend,
        marker_color=BACKEND_COLORS[backend],
        customdata=subset["Workflow ID"],
        hovertemplate="%{customdata}<br>Edges: %{x:.1f}×1M<br>Time: %{y:.1f} min<extra></extra>",
    ))

fig.update_layout(
    title="Base Graph Creation — by Backend",
    xaxis_title="Edge Count (×1M)",
    yaxis_title="Time (minutes)",
    height=600,
    template=PLOTLY_THEME,
)
fig.show()

## Figure 2 — Fine/H3 Graph Creation
Time to create the fine or H3 graph, grouped by Backend × Graph Mode.

In [3]:
y_col = "Fine/H3 Graph Creation (min)"
valid = df.dropna(subset=[y_col])

fig = go.Figure()
for backend in ["PostGIS", "GeoPackage"]:
    for mode in ["FINE", "H3"]:
        subset = valid[(valid["Backend"] == backend) & (valid["Graph Mode"] == mode)].sort_values("edges_per_1M")
        if subset.empty:
            continue
        label = f"{backend} + {mode}"
        fig.add_trace(go.Scatter(
            x=subset["edges_per_1M"],
            y=subset[y_col],
            mode="markers",
            name=label,
            line=dict(color=BACKEND_COLORS[backend], dash=MODE_DASH[mode]),
            customdata=subset["Workflow ID"],
            hovertemplate="%{customdata}<br>Edges: %{x:.1f}×1M<br>Time: %{y:.1f} min<extra></extra>",
        ))

fig.update_layout(
    title="Fine/H3 Graph Creation — by Backend × Graph Mode",
    xaxis_title="Edge Count (×1M)",
    yaxis_title="Time (minutes)",
    height=600,
    template=PLOTLY_THEME,
)
fig.show()

## Figure 3 — Weighting & Enrichment
Time for weighting and enrichment, by Backend. X axis uses 2× edge count (weighting processes directed edges).

In [4]:
y_col = "Weighting & Enrichment (min)"
valid = df.dropna(subset=[y_col])

fig = go.Figure()
for backend in ["PostGIS", "GeoPackage"]:
    subset = valid[valid["Backend"] == backend].sort_values("edges_2x_per_1M")
    fig.add_trace(go.Scatter(
        x=subset["edges_2x_per_1M"],
        y=subset[y_col],
        mode="markers",
        name=backend,
        marker_color=BACKEND_COLORS[backend],
        customdata=subset["Workflow ID"],
        hovertemplate="%{customdata}<br>Edges (2×): %{x:.1f}×1M<br>Time: %{y:.1f} min<extra></extra>",
    ))

fig.update_layout(
    title="Weighting & Enrichment — by Backend",
    xaxis_title="Directed Edge Count (×1M)",
    yaxis_title="Time (minutes)",
    height=600,
    template=PLOTLY_THEME,
)
fig.show()

## Figure 4 — Pathfinding & Export
Time for pathfinding and export, by Backend. X axis is edge count — A* explores edges during route search.

In [5]:
y_col = "Pathfinding & Export (min)"
valid = df.dropna(subset=[y_col])

fig = go.Figure()
for backend in ["PostGIS", "GeoPackage"]:
    subset = valid[valid["Backend"] == backend].sort_values("edges_2x_per_1M")
    fig.add_trace(go.Scatter(
        x=subset["edges_2x_per_1M"],
        y=subset[y_col],
        mode="markers",
        name=backend,
        marker_color=BACKEND_COLORS[backend],
        customdata=subset["Workflow ID"],
        hovertemplate="%{customdata}<br>Edges (2×): %{x:.1f}×1M<br>Time: %{y:.1f} min<extra></extra>",
    ))

fig.update_layout(
    title="Pathfinding & Export — by Backend",
    xaxis_title="Directed Edge Count (×1M)",
    yaxis_title="Time (minutes)",
    height=600,
    template=PLOTLY_THEME,
)
fig.show()

## Figure 5 — Total Time
End-to-end pipeline time, grouped by Backend × Graph Mode.

In [6]:
y_col = "Total Time (min)"
valid = df.dropna(subset=[y_col])

fig = go.Figure()
for backend in ["PostGIS", "GeoPackage"]:
    for mode in ["FINE", "H3"]:
        subset = valid[(valid["Backend"] == backend) & (valid["Graph Mode"] == mode)].sort_values("edges_2x_per_1M")
        if subset.empty:
            continue
        label = f"{backend} + {mode}"
        fig.add_trace(go.Scatter(
            x=subset["edges_2x_per_1M"],
            y=subset[y_col],
            mode="markers",
            name=label,
            line=dict(color=BACKEND_COLORS[backend], dash=MODE_DASH[mode]),
            customdata=subset["Workflow ID"],
            hovertemplate="%{customdata}<br>Edges (2×): %{x:.1f}×1M<br>Time: %{y:.1f} min<extra></extra>",
        ))

fig.update_layout(
    title="Total Pipeline Time — by Backend × Graph Mode",
    xaxis_title="Directed Edge Count (×1M)",
    yaxis_title="Time (minutes)",
    height=600,
    template=PLOTLY_THEME,
)
fig.show()